## **EDA**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import lag_plot
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
ruta_train = Path("../data/2.processed/processed_train.csv").resolve()
ruta_test = Path("../data/2.processed/processed_test.csv").resolve()
df_train = pd.read_csv(ruta_train)
df_test = pd.read_csv(ruta_test)

In [ ]:
df_train['Date'] = pd.to_datetime(df_train['Date'], format='%Y-%m-%d')
df_test['Date'] = pd.to_datetime(df_test['Date'], format='%Y-%m-%d')

def add_date_features(df):
    df['day'] = df['Date'].dt.day
    df['month'] = df['Date'].dt.month
    df['year'] = df['Date'].dt.year
    df['bimester'] = (df['Date'].dt.month - 1) // 2 + 1  # 1 a 6
    df['quarter'] = df['Date'].dt.quarter               # 1 a 4 - trimestre
    df['semester'] = (df['Date'].dt.month - 1) // 6 + 1 # 1 a 2 - semestre
    return df

# 2. Aplicalo a los dos
df_train = add_date_features(df_train)
df_test = add_date_features(df_test)

In [ ]:
sns.set_style("whitegrid")

# 1. Solo las numéricas vs Weekly_Sales
cols_numericas = ['Size','Fuel_Price','CPI','Unemployment','Temperature']
df_plot = df_train # tu df ya procesado

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cols_numericas):
    sns.scatterplot(data=df_plot, x=col, y='Weekly_Sales', alpha=0.3, s=15, ax=axes[i])
    axes[i].set_title(f'{col} vs Weekly_Sales')

# Quitamos el subplot vacío
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cols_numericas = ['Size','Fuel_Price','CPI','Unemployment','Temperature', 'Weekly_Sales']

# Calculamos correlación
corr = df_train[cols_numericas].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            center=0,
            linewidths=0.5,
            square=True)

plt.title('Mapa de calor - Correlación con Weekly_Sales')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Promedio de ventas por tienda
sns.barplot(data=df_plot.groupby('Store')['Weekly_Sales'].mean().reset_index(),
            x='Store', y='Weekly_Sales', ax=axes[0])
axes[0].set_title('Promedio Weekly_Sales por Store')

# Promedio por Dept - top 20 para que se vea
dept_mean = df_plot.groupby('Dept')['Weekly_Sales'].mean().sort_values(ascending=False).head(20)
sns.barplot(x=dept_mean.index, y=dept_mean.values, ax=axes[1])
axes[1].set_title('Top 20 Dept con más Weekly_Sales')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Agrupamos por fecha: sumamos las ventas y marcamos si ese dia fue feriado
df_daily = df_train.groupby('Date').agg(
    Weekly_Sales=('Weekly_Sales', 'sum'),
    IsHoliday=('IsHoliday', 'max')  # si al menos una tienda fue True, es feriado
).reset_index()

# 2. Gráfico pro
plt.figure(figsize=(14, 6))

# Línea de ventas
sns.lineplot(data=df_daily, x='Date', y='Weekly_Sales', color='steelblue', label='Weekly_Sales')

# Puntos rojos solo donde IsHoliday = True
holidays = df_daily[df_daily['IsHoliday'] == True]
sns.scatterplot(data=holidays, x='Date', y='Weekly_Sales', color='red', s=100, label='IsHoliday = True', zorder=5)

plt.title('Weekly_Sales en el tiempo con Holidays marcados')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1. Ranking de las 4 tiendas con más ventas totales
top_4_stores = df_train.groupby('Store')['Weekly_Sales'].sum().nlargest(4).index.tolist()
print("Top 4 tiendas:", top_4_stores)

# 2. Filtramos solo esas 4 y agrupamos por fecha para que no se promedie
df_top4 = df_train[df_train['Store'].isin(top_4_stores)]
df_top4_daily = df_top4.groupby(['Date', 'Store'])['Weekly_Sales'].sum().reset_index()

# 3. Gráfico solo de las 4
plt.figure(figsize=(14, 6))
sns.lineplot(data=df_top4_daily, x='Date', y='Weekly_Sales', hue='Store', linewidth=2)

plt.title('Top 4 Stores por Weekly_Sales')
plt.xticks(rotation=45)
plt.legend(title='Store')
plt.tight_layout()
plt.show()

In [ ]:
top_stores = df_train.groupby('Store')['Weekly_Sales'].sum().nlargest(8).index
top_depts = df_train.groupby('Dept')['Weekly_Sales'].sum().nlargest(8).index

df_filt = df_train[df_train['Store'].isin(top_stores) & df_train['Dept'].isin(top_depts)]

pivot = df_filt.groupby(['Dept','Store'])['Weekly_Sales'].sum().reset_index().pivot(index='Dept', columns='Store', values='Weekly_Sales')

plt.figure(figsize=(12,8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu', linewidths=.5)
plt.title('Heatmap: Ventas Totales - Top Stores vs Top Depts')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

X = df_train[['Size','Fuel_Price','CPI','Unemployment','Temperature','Store','Dept','month','quarter','IsHoliday']]
y = df_train['Weekly_Sales']

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance)

In [ ]:
markdown_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']

def clean_markdowns(df):
    # 1. Flag de si hubo promo
    for col in markdown_cols:
        if col in df.columns:
            df[f'{col}_active'] = df[col].notna().astype(int)

    # 2. NA = no hubo promo = 0
    df[markdown_cols] = df[markdown_cols].fillna(0)

    # 3. Features útiles
    df['MarkDown_Total'] = df[markdown_cols].sum(axis=1)
    df['MarkDown_Count'] = (df[markdown_cols] > 0).sum(axis=1)
    return df

df_train = clean_markdowns(df_train)
df_test = clean_markdowns(df_test)

print("Train:", df_train[markdown_cols].isna().sum().sum(), "NaNs restantes")
print("Test:", df_test[markdown_cols].isna().sum().sum(), "NaNs restantes")

In [ ]:
# Asumo que ya corriste src/make_final_dataset.py
df_train['Date'] = pd.to_datetime(df_train['Date'])
df_train = df_train.sort_values(['Store','Dept','Date'])

df_train['lag_1'] = df_train.groupby(['Store','Dept'])['Weekly_Sales'].shift(1)
df_train['lag_4'] = df_train.groupby(['Store','Dept'])['Weekly_Sales'].shift(4)
df_train['lag_52'] = df_train.groupby(['Store','Dept'])['Weekly_Sales'].shift(52)

# Filtra 1 Store-Dept para que no sea ruido, ej: Store 1, Dept 1
sample = df_train[(df_train['Store']==1) & (df_train['Dept']==1)].dropna()

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4))
axes[0].scatter(sample['lag_1'], sample['Weekly_Sales'], alpha=0.6)
axes[0].set_title('lag_1 vs Actual - Correlación semanal')
axes[1].scatter(sample['lag_4'], sample['Weekly_Sales'], alpha=0.6)
axes[1].set_title('lag_4 vs Actual - Mensual')
axes[2].scatter(sample['lag_52'], sample['Weekly_Sales'], alpha=0.6)
axes[2].set_title('lag_52 vs Actual - Anual (Navidad)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(sample['Date'], sample['Weekly_Sales'], label='Real', linewidth=2)
plt.plot(sample['Date'], sample['lag_1'], label='lag_1', alpha=0.7)
plt.plot(sample['Date'], sample['lag_52'], label='lag_52', alpha=0.7)
plt.legend()
plt.title('Store 1 Dept 1 - Real vs Lags')
plt.show()

In [ ]:
corr = sample[['Weekly_Sales','lag_1','lag_4','lag_52','Size','month','MarkDown_Total']].corr()
sns.heatmap(corr, annot=True, cmap='Blues')
plt.title('Correlación - Lags')
plt.show()

In [ ]:
sample['error_lag52'] = sample['Weekly_Sales'] - sample['lag_52']
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
sns.histplot(sample['error_lag52'], kde=True)
plt.title('Distribución Error lag_52')
plt.subplot(1,2,2)
plt.scatter(sample['Date'], sample['error_lag52'])
plt.title('Error lag_52 en el tiempo - picos = Navidad')
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(figsize=(10,4))
plot_acf(sample['Weekly_Sales'], lags=60, ax=ax)
ax.set_title('ACF Store 1 Dept 1 - Picos en 1, 4 y 52')
plt.show()